In [4]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
from datasets import load_dataset

from src.model_loader import load_model
from src.intervention import (
    load_direction, gap_from_metadata,
    score_dataset, run_recovery, recovery_table,
    _make_ablation_fn,
)
from src.layer_sweep import run_layer_sweep

CHECKPOINTS = {
    "GradDiff": "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_GradDiff_lr1e-05_alpha5_epoch5",
    "NPO":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_NPO_lr1e-05_beta0.5_alpha1_epoch10",
    "AltPO":    "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_AltPO_lr5e-05_beta0.1_alpha1_epoch10",
    "SimNPO":   "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_SimNPO_lr2e-05_b4.5_a1_d1_g0.125_ep10",
    "RMU":      "open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_RMU_lr5e-05_layer10_scoeff10_epoch10",
}
RETAIN = "open-unlearning/tofu_Llama-3.2-1B-Instruct_retain90"
BASE_MODEL = "open-unlearning/tofu_Llama-3.2-1B-Instruct_full"

# Start with RMU at its training layer.
LAYER      = 10
LAYER_NAME = f"model.layers.{LAYER}"

In [5]:
# forget10 is the split your checkpoints were unlearned on.
tofu = load_dataset("locuslab/TOFU", "forget10")["train"]

forget_prompts = [ex["question"] for ex in tofu]
forget_answers = [ex["answer"]   for ex in tofu]

print(f"{len(forget_prompts)} forget pairs")
print("example Q:", forget_prompts[0])
print("example A:", forget_answers[0])

400 forget pairs
example Q: What is the full name of the author born in Taipei, Taiwan on 05/11/1991 who writes in the genre of leadership?
example A: The author's full name is Hsiao Yun-Hwa.


In [6]:
# The sweep needs retain questions too (forget vs retain is what the probe separates).
retain_tofu    = load_dataset("locuslab/TOFU", "retain90")["train"]
retain_prompts = [ex["question"] for ex in retain_tofu]

# model/tok/dev for RMU must be loaded first (Cell 3's load_model line),
# OR load here if you haven't yet:
for model_key in CHECKPOINTS.keys():
    # Direction + metadata must come from a sweep over THIS checkpoint's activations.
    DIR_PATH  = f"../data/sweep_{model_key.lower()}/layer{LAYER}/direction.npy"
    META_PATH = f"../results/sweep_{model_key.lower()}/sweep_metadata.json"

    data_dir = f"../data/sweep_{model_key.lower()}/"
    if os.path.exists(data_dir) and os.path.isdir(data_dir):
        if len(os.listdir(data_dir)) > 0:
            continue
    model, tok, dev = load_model(CHECKPOINTS[model_key])

    metrics = run_layer_sweep(
        model, tok,
        forget_questions=forget_prompts,
        retain_questions=retain_prompts,
        device=dev,
        n_layers=16,
        data_dir=f"../data/sweep_{model_key.lower()}",
        results_dir=f"../results/sweep_{model_key.lower()}",
        model_id=CHECKPOINTS[model_key],
    )

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Loaded: open-unlearning/unlearn_tofu_Llama-3.2-1B-Instruct_forget10_GradDiff_lr1e-05_alpha5_epoch5
Device: mps | dtype: torch.float16
Params: 1.2B


Layer sweep:  25%|██▌       | 4/16 [00:00<00:00, 14.66it/s]

layer  0 | train 1.000 | test 0.838 | gap 0.013 | overlap 0.455
layer  1 | train 1.000 | test 0.762 | gap 0.030 | overlap 0.887
layer  2 | train 1.000 | test 0.725 | gap 0.049 | overlap 0.682
layer  3 | train 1.000 | test 0.781 | gap 0.097 | overlap 0.915


Layer sweep:  50%|█████     | 8/16 [00:00<00:00, 16.46it/s]

layer  4 | train 1.000 | test 0.781 | gap 0.126 | overlap 0.838
layer  5 | train 1.000 | test 0.738 | gap 0.149 | overlap 0.825
layer  6 | train 1.000 | test 0.756 | gap 0.151 | overlap 0.790
layer  7 | train 1.000 | test 0.756 | gap 0.165 | overlap 0.830


Layer sweep:  81%|████████▏ | 13/16 [00:00<00:00, 18.58it/s]

layer  8 | train 1.000 | test 0.762 | gap 0.176 | overlap 0.835
layer  9 | train 1.000 | test 0.800 | gap 0.228 | overlap 0.823
layer 10 | train 1.000 | test 0.875 | gap 0.254 | overlap 0.482
layer 11 | train 1.000 | test 0.863 | gap 0.337 | overlap 0.152
layer 12 | train 1.000 | test 0.887 | gap 0.447 | overlap 0.365


Layer sweep: 100%|██████████| 16/16 [00:00<00:00, 18.45it/s]

layer 13 | train 1.000 | test 0.887 | gap 0.614 | overlap 0.693
layer 14 | train 1.000 | test 0.956 | gap 1.178 | overlap 0.022
layer 15 | train 1.000 | test 0.969 | gap 1.976 | overlap 0.010

Best layer by test accuracy: 15 (0.969)
Raw data saved to ../data/sweep_graddiff/
Artifacts saved to ../results/sweep_graddiff/
